# Feature-set Ablation Notebook (Singletons vs Cumulative Ladder)

Goal: test whether *individual* feature groups (F2-only, F3-only, etc.) have better edge than the default cumulative ladder (F1, F1+F2, ..., F5).

This notebook runs the pipeline **with a custom `config.feature_sets`** without permanently changing `src/config.py`.

Outputs will be written under `outputs/` using the notebook tag you set (default: `06`).


In [1]:
%matplotlib inline
import pandas as pd
from src.config import AppConfig
from src.pipeline import run_pipeline

config = AppConfig()
config.horizons = [1, 3]
config.top_k = 5
config.transaction_cost_bps = 10

print('Default feature sets:', list(config.feature_sets.keys()))

Default feature sets: ['F1_momentum', 'F2_momentum_reversal', 'F3_plus_risk_liquidity', 'F4_plus_cross_sectional', 'F5_full_finance_no_fundamental']


## 1) Define custom feature sets
We keep two families:
- **Cumulative ladder**: the existing F1→F5 approach (baseline)
- **Singletons**: each group alone (e.g., reversal-only, macro-only)

Note: You currently excluded `fundamental` from F5 due to data quality, so this notebook follows that default.

In [3]:
# Cumulative ladder (same as config default, but explicitly defined here)
ladder = {
    'F1_momentum': ['momentum'],
    'F2_momentum_reversal': ['momentum','reversal'],
    'F3_plus_risk_liquidity': ['momentum','reversal','volatility','liquidity'],
    'F4_plus_cross_sectional': ['momentum','reversal','volatility','liquidity','cross_sectional'],
    'F5_full_finance_no_fundamental': ['momentum','reversal','volatility','liquidity','cross_sectional','macro'],
}

# Single-group ('lonely') feature sets
singletons = {
    'S1_momentum_only': ['momentum'],
    'S2_reversal_only': ['reversal'],
    'S3_volatility_only': ['volatility'],
    'S4_liquidity_only': ['liquidity'],
    'S5_cross_sectional_only': ['cross_sectional'],
    'S6_macro_only': ['macro'],
}

# Combine them into one experiment list
config.feature_sets = {**ladder, **singletons}
print('Running feature sets:', len(config.feature_sets))
list(config.feature_sets.keys())

Running feature sets: 11


['F1_momentum',
 'F2_momentum_reversal',
 'F3_plus_risk_liquidity',
 'F4_plus_cross_sectional',
 'F5_full_finance_no_fundamental',
 'S1_momentum_only',
 'S2_reversal_only',
 'S3_volatility_only',
 'S4_liquidity_only',
 'S5_cross_sectional_only',
 'S6_macro_only']

## 2) Run pipeline (choose mode)
- Use `mode='synthetic'` to check everything works fast
- Use `mode='live'` for real results (slower, depends on Yahoo Finance)

In [4]:
notebook_tag = '06'
mode = 'live'  # change to 'synthetic' for fast debug

paths = run_pipeline(notebook_tag=notebook_tag, mode=mode, config=config)
paths

{'metrics': WindowsPath('C:/Users/user/Downloads/Moltbot/HKU-FYP/fyp_finance_ml_v2/outputs/metrics/06_live_metrics.csv'),
 'backtest': WindowsPath('C:/Users/user/Downloads/Moltbot/HKU-FYP/fyp_finance_ml_v2/outputs/metrics/06_live_backtest_summary.csv'),
 'relative': WindowsPath('C:/Users/user/Downloads/Moltbot/HKU-FYP/fyp_finance_ml_v2/outputs/metrics/06_live_relative_summary.csv'),
 'decile': WindowsPath('C:/Users/user/Downloads/Moltbot/HKU-FYP/fyp_finance_ml_v2/outputs/tables/06_live_bucket_returns.csv'),
 'data_quality': WindowsPath('C:/Users/user/Downloads/Moltbot/HKU-FYP/fyp_finance_ml_v2/outputs/tables/06_live_data_quality.csv'),
 'metadata': WindowsPath('C:/Users/user/Downloads/Moltbot/HKU-FYP/fyp_finance_ml_v2/outputs/metadata/06_live_run_metadata.json'),
 'summary_figure': WindowsPath('C:/Users/user/Downloads/Moltbot/HKU-FYP/fyp_finance_ml_v2/outputs/figures/06_live_four_panel_summary.png'),
 'curves_dir': WindowsPath('C:/Users/user/Downloads/Moltbot/HKU-FYP/fyp_finance_ml_v2/

## 3) Load outputs and compare singletons vs ladder
We compare at two levels:
- signal quality: Rank IC
- economic performance: Sharpe / max drawdown

Backtest tables include two execution modes: `open_to_open` and `close_to_close`.

In [5]:
metrics = pd.read_csv(paths['metrics'])
bt = pd.read_csv(paths['backtest'])

print('metrics rows:', len(metrics))
print('backtest rows:', len(bt))
metrics.head()

metrics rows: 44
backtest rows: 88


,notebook_tag,mode,feature_set,feature_groups,model,n_features_used,horizon_days,threshold,validation_balanced_accuracy,always_up_bal_acc,...,fn,tp,positive_rate,avg_score,roc_auc,rank_ic,icir,top_k_hit_rate,top_bottom_spread,bucket_monotonicity
0,6,live,F1_momentum,momentum,logistic_regression,14,1,0.48,0.501010,0.5,...,2626,29186,0.524112,0.499945,0.510358,0.009852,0.047633,0.535809,-0.000693,0.75
1,6,live,F1_momentum,momentum,random_forest,14,1,0.46,0.501603,0.5,...,667,31145,0.524112,0.500037,0.507014,0.007519,0.039429,0.540584,0.000735,0.50
2,6,live,F2_momentum_reversal,"momentum,reversal",logistic_regression,21,1,0.48,0.501614,0.5,...,3520,28292,0.524112,0.499982,0.507234,0.005092,0.025302,0.543236,0.000571,0.75
3,6,live,F2_momentum_reversal,"momentum,reversal",random_forest,21,1,0.47,0.501198,0.5,...,1365,30447,0.524112,0.500127,0.507812,0.007643,0.040616,0.533687,0.000159,0.50
4,6,live,F3_plus_risk_liquidity,"momentum,reversal,volatility,liquidity",logistic_regression,33,1,0.45,0.500331,0.5,...,538,31274,0.524112,0.499041,0.506710,0.007478,0.038305,0.545889,0.001341,0.50


In [6]:
# Join metrics onto backtest for convenient sorting
key = ['feature_set','model','horizon_days']
df = bt.merge(metrics[['feature_set','model','horizon_days','balanced_accuracy','rank_ic','icir']], on=key, how='left')

# Add a label for singleton vs ladder
df['family'] = df['feature_set'].apply(lambda s: 'singleton' if s.startswith('S') else 'ladder')

df[['feature_set','family','model','horizon_days','execution_mode','sharpe','max_drawdown','rank_ic','balanced_accuracy']].head()

,feature_set,family,model,horizon_days,execution_mode,sharpe,max_drawdown,rank_ic,balanced_accuracy
0,F1_momentum,ladder,logistic_regression,1,open_to_open,-1.027978,-0.472713,0.009852,0.505308
1,F1_momentum,ladder,logistic_regression,1,close_to_close,-0.890199,-0.488804,0.009852,0.505308
2,F1_momentum,ladder,random_forest,1,open_to_open,-1.150226,-0.414022,0.007519,0.502586
3,F1_momentum,ladder,random_forest,1,close_to_close,-0.609201,-0.312859,0.007519,0.502586
4,F2_momentum_reversal,ladder,logistic_regression,1,open_to_open,-0.819393,-0.376245,0.005092,0.505502


In [7]:
# Top results by Sharpe for each horizon + execution_mode
for h in sorted(df.horizon_days.unique()):
    for em in sorted(df.execution_mode.unique()):
        sub = df[(df.horizon_days==h) & (df.execution_mode==em)].sort_values('sharpe', ascending=False)
        display(sub[['feature_set','family','model','sharpe','annualized_return','max_drawdown','avg_turnover','avg_cost_drag','rank_ic','balanced_accuracy']].head(10))


,feature_set,family,model,sharpe,annualized_return,max_drawdown,avg_turnover,avg_cost_drag,rank_ic,balanced_accuracy
41,S6_macro_only,singleton,logistic_regression,0.614596,0.098343,-0.219262,0.000000,0.000000,NaN,0.498331
43,S6_macro_only,singleton,random_forest,0.614596,0.098343,-0.219262,0.000000,0.000000,NaN,0.494439
17,F5_full_finance_no_fundamental,ladder,logistic_regression,0.324389,0.053385,-0.188887,1.319149,0.001319,0.007104,0.502729
29,S3_volatility_only,singleton,logistic_regression,0.179173,0.016171,-0.125560,0.908511,0.000909,0.012992,0.500439
37,S5_cross_sectional_only,singleton,logistic_regression,0.158343,0.006859,-0.150436,1.235106,0.001235,0.011139,0.503985
13,F4_plus_cross_sectional,ladder,logistic_regression,0.152332,0.006207,-0.237520,1.276596,0.001277,0.009005,0.501135
25,S2_reversal_only,singleton,logistic_regression,-0.078357,-0.048657,-0.282632,1.659574,0.001660,0.012961,0.503612
33,S4_liquidity_only,singleton,logistic_regression,-0.188978,-0.102817,-0.378094,1.203191,0.001203,0.014900,0.500945
9,F3_plus_risk_liquidity,ladder,logistic_regression,-0.423102,-0.155447,-0.416559,1.611702,0.001612,0.007478,0.504423
7,F2_momentum_reversal,ladder,random_forest,-0.564508,-0.152450,-0.289386,1.578723,0.001579,0.007643,0.503749


,feature_set,family,model,sharpe,annualized_return,max_drawdown,avg_turnover,avg_cost_drag,rank_ic,balanced_accuracy
12,F4_plus_cross_sectional,ladder,logistic_regression,0.849264,0.225976,-0.236580,1.275733,0.001276,0.009005,0.501135
40,S6_macro_only,singleton,logistic_regression,0.620715,0.101822,-0.227221,0.000000,0.000000,NaN,0.498331
42,S6_macro_only,singleton,random_forest,0.620715,0.101822,-0.227221,0.000000,0.000000,NaN,0.494439
28,S3_volatility_only,singleton,logistic_regression,0.536753,0.091162,-0.152515,0.909867,0.000910,0.012992,0.500439
16,F5_full_finance_no_fundamental,ladder,logistic_regression,0.218705,0.021825,-0.248350,1.318400,0.001318,0.007104,0.502729
36,S5_cross_sectional_only,singleton,logistic_regression,0.058484,-0.023615,-0.172739,1.235200,0.001235,0.011139,0.503985
8,F3_plus_risk_liquidity,ladder,logistic_regression,-0.405466,-0.158663,-0.305030,1.613867,0.001614,0.007478,0.504423
38,S5_cross_sectional_only,singleton,random_forest,-0.545213,-0.144547,-0.249719,1.682133,0.001682,0.011720,0.503867
32,S4_liquidity_only,singleton,logistic_regression,-0.584307,-0.215135,-0.404200,1.201067,0.001201,0.014900,0.500945
14,F4_plus_cross_sectional,ladder,random_forest,-0.606996,-0.167379,-0.261188,1.649067,0.001649,0.011681,0.507851


,feature_set,family,model,sharpe,annualized_return,max_drawdown,avg_turnover,avg_cost_drag,rank_ic,balanced_accuracy
47,F1_momentum,ladder,random_forest,0.870851,0.134216,-0.205143,0.864171,0.000864,0.007283,0.508723
67,S1_momentum_only,singleton,random_forest,0.870851,0.134216,-0.205143,0.864171,0.000864,0.007283,0.508723
87,S6_macro_only,singleton,random_forest,0.807912,0.080236,-0.204261,0.000000,0.000000,NaN,0.484528
85,S6_macro_only,singleton,logistic_regression,0.780122,0.076905,-0.206752,0.006417,0.000006,-0.007655,0.497215
51,F2_momentum_reversal,ladder,random_forest,0.741146,0.112807,-0.224759,0.858824,0.000859,0.006604,0.509397
55,F3_plus_risk_liquidity,ladder,random_forest,0.200666,0.019379,-0.203258,1.051337,0.001051,0.011161,0.504626
73,S3_volatility_only,singleton,logistic_regression,0.097717,0.003785,-0.179742,1.136898,0.001137,0.022591,0.500280
57,F4_plus_cross_sectional,ladder,logistic_regression,-0.180676,-0.040889,-0.314443,1.186096,0.001186,0.004662,0.503284
53,F3_plus_risk_liquidity,ladder,logistic_regression,-0.234058,-0.055350,-0.343906,1.034225,0.001034,-0.010469,0.502722
77,S4_liquidity_only,singleton,logistic_regression,-0.410342,-0.075344,-0.264013,1.022460,0.001022,0.017186,0.500940


,feature_set,family,model,sharpe,annualized_return,max_drawdown,avg_turnover,avg_cost_drag,rank_ic,balanced_accuracy
46,F1_momentum,ladder,random_forest,1.180293,0.190581,-0.205505,0.865416,0.000865,0.007283,0.508723
66,S1_momentum_only,singleton,random_forest,1.180293,0.190581,-0.205505,0.865416,0.000865,0.007283,0.508723
50,F2_momentum_reversal,ladder,random_forest,0.979776,0.157169,-0.226542,0.860054,0.000860,0.006604,0.509397
86,S6_macro_only,singleton,random_forest,0.791783,0.079235,-0.208443,0.000000,0.000000,NaN,0.484528
84,S6_macro_only,singleton,logistic_regression,0.784530,0.078296,-0.209276,0.006434,0.000006,-0.007655,0.497215
54,F3_plus_risk_liquidity,ladder,random_forest,0.473459,0.065672,-0.193460,1.049866,0.001050,0.011161,0.504626
56,F4_plus_cross_sectional,ladder,logistic_regression,0.048112,-0.005378,-0.330892,1.184987,0.001185,0.004662,0.503284
52,F3_plus_risk_liquidity,ladder,logistic_regression,0.012096,-0.013716,-0.341417,1.034853,0.001035,-0.010469,0.502722
72,S3_volatility_only,singleton,logistic_regression,-0.073214,-0.021108,-0.197076,1.136729,0.001137,0.022591,0.500280
76,S4_liquidity_only,singleton,logistic_regression,-0.282595,-0.057906,-0.227122,1.024129,0.001024,0.017186,0.500940


In [ ]:
# Compare ladder vs singleton averages
summary = df.groupby(['family','horizon_days','execution_mode']).agg(
    n=('sharpe','count'),
    sharpe_mean=('sharpe','mean'),
    sharpe_max=('sharpe','max'),
    rank_ic_mean=('rank_ic','mean'),
    balacc_mean=('balanced_accuracy','mean'),
).reset_index()
summary.sort_values(['horizon_days','execution_mode','family'])